# Fast ℓp Proximity Operator — Phase 1
## GitHub-first, accuracy-matched GPU benchmark archive

**Start here.** Run cells in order. Section A automatically discovers the public archive in a local checkout **or clones** the `revision/major-experiments` branch from [Fast_Lp_Prox](https://github.com/Jy-U1/Fast_Lp_Prox). **No manual ZIP upload is needed.** Sections A–D analyze recorded results without a GPU.

The public archive contains six **formal** A100 sweeps ($10^6$ inputs, float32, 30 timing repeats), two **pilot** sweeps ($10^4$ inputs), and three **smoke** tests. Derived tables, figures, QA reports, and the recorded SHA-256 checksums are reproducible.

**Reproducibility boundaries**
- Results are sourced from archived, unchanged raw CSVs; analysis runs in a **temporary working directory**, leaving the Git checkout clean.
- We check all archived CSV SHA-256 hashes before analysis and keep formal/pilot/smoke separate.
- The formal *benchmark code* used recorded Git commit `7a1e32b`; the *data archive* lives on branch `revision/major-experiments`, whose current checkout commit is reported at runtime.
- Medians are **online times** only. Chebyshev coefficient fitting and reference construction are excluded.
- Maximum normalized error is empirical on the sampled input set, **not** a certified whole-domain bound. Stationarity residual is a distinct, unnormalized metric.
- **Section E is opt-in**: it checks out benchmark commit `7a1e32b` and writes fresh GPU CSVs to `data/rerun/`, never over the archived data.
- Neither this notebook nor the ZIP export includes reviewer correspondence or private paper drafts.

**Workflow:** A (get public data) → B (rebuild/audit) → C (figures) → D (explore) → optional E (GPU rerun) → F (download ZIP).


---
## A · Discover or clone the public Phase 1 archive

In a normal Colab runtime, this section automatically downloads the public branch from GitHub. In a local VS Code/Jupyter checkout, it finds `revision/phase1/` without a download. To explicitly refresh from GitHub, set `SOURCE_MODE = 'github'`; otherwise keep the default `'auto'`.

The archive is **copied to an isolated workspace** before analysis, so rebuilding tables and figures will not modify version-controlled files. The SHA-256 values of **all 11 original CSVs** are checked against `manifest.json`.


In [ ]:
from pathlib import Path
import os, sys, json, zipfile, subprocess, shutil, tempfile, hashlib
import pandas as pd
from IPython.display import display, Image

REPO_URL = 'https://github.com/Jy-U1/Fast_Lp_Prox.git'
REPO_BRANCH = 'revision/major-experiments'
SOURCE_MODE = 'auto'    # 'auto' = prefer local archive, else clone; 'github' = fetch public branch
ARCHIVE_RELATIVE_DIR = Path('revision/phase1')
PUBLIC_REPO = (Path('/content/Fast_Lp_Prox_Phase1_Public') if Path('/content').is_dir()
               else Path.home() / '.cache' / 'Fast_Lp_Prox_Phase1_Public')

def is_archive_folder(path):
    return (path / 'data' / 'formal').is_dir() and (path / 'manifest.json').is_file() \
           and (path / 'scripts' / 'rebuild_phase1_artifacts.py').is_file()

def local_archive_candidates():
    cwd = Path.cwd().resolve()
    bases = [cwd, *cwd.parents, Path('/content/Fast_Lp_Prox'), PUBLIC_REPO]
    for base in bases:
        yield base
        yield base / ARCHIVE_RELATIVE_DIR

def load_public_archive():
    if SOURCE_MODE not in {'auto', 'github'}:
        raise ValueError("SOURCE_MODE must be 'auto' or 'github'")
    if SOURCE_MODE == 'auto':
        for candidate in local_archive_candidates():
            if is_archive_folder(candidate):
                return candidate.resolve(), 'existing local archive'

    PUBLIC_REPO.parent.mkdir(parents=True, exist_ok=True)
    if (PUBLIC_REPO / '.git').is_dir():
        # Safe: only the dedicated public-data checkout is refreshed.
        subprocess.run(['git', '-C', str(PUBLIC_REPO), 'checkout', REPO_BRANCH], check=True)
        subprocess.run(['git', '-C', str(PUBLIC_REPO), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
    elif PUBLIC_REPO.exists():
        raise RuntimeError(f'Expected a Git repository at {PUBLIC_REPO}; move this folder aside first.')
    else:
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH,
                        REPO_URL, str(PUBLIC_REPO)], check=True)
    archive = PUBLIC_REPO / ARCHIVE_RELATIVE_DIR
    if not is_archive_folder(archive):
        raise FileNotFoundError(f'Phase 1 data not found on {REPO_BRANCH}: {archive}')
    return archive.resolve(), 'public GitHub branch'

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def check_archive_integrity(archive):
    manifest = json.loads((archive / 'manifest.json').read_text(encoding='utf-8'))
    entries = manifest.get('raw_csv_files', [])
    if len(entries) != 11:
        raise ValueError(f'Expected 11 source CSV checksums; found {len(entries)}')
    for entry in entries:
        rel = Path(entry['relative_path'])
        if rel.is_absolute() or '..' in rel.parts:
            raise ValueError(f'Unsafe archive path in manifest: {rel}')
        path = archive / rel
        if not path.is_file():
            raise FileNotFoundError(f'Missing recorded CSV: {path}')
        actual = sha256_file(path)
        if actual != entry['sha256']:
            raise RuntimeError(f'SHA-256 mismatch for {rel}: {actual} != {entry["sha256"]}')
    return manifest, len(entries)

ARCHIVE_ROOT, archive_source = load_public_archive()
archive_manifest, hash_count = check_archive_integrity(ARCHIVE_ROOT)
ROOT = Path(tempfile.mkdtemp(prefix='FastLp_Phase1_Workspace_')).resolve()
shutil.copytree(ARCHIVE_ROOT, ROOT, dirs_exist_ok=True)
FORMAL_DIR = ROOT / 'data' / 'formal'

archive_git_revision = None
proc = subprocess.run(['git', '-C', str(ARCHIVE_ROOT), 'rev-parse', 'HEAD'],
                      capture_output=True, text=True)
if proc.returncode == 0:
    archive_git_revision = proc.stdout.strip()

print('Data source:', archive_source)
print('Public data directory:', ARCHIVE_ROOT)
print('Archive Git checkout:', archive_git_revision or '(standalone archive)')
print('Recorded experiment code commit:', archive_manifest['formal_code_commit'])
print('Raw CSV SHA-256 checks passed:', hash_count, 'of', hash_count)
print('Isolated analysis workspace:', ROOT)
print('Formal / pilot / smoke CSV files:',
      len(list(FORMAL_DIR.glob('*.csv'))),
      len(list((ROOT / 'data' / 'pilot').glob('*.csv'))),
      len(list((ROOT / 'data' / 'smoke').glob('*.csv'))))


## B · Rebuild figures, tables and quality reports from archived CSVs

This invokes the archived analysis script **inside the isolated workspace**. It does not use a GPU or modify any raw experimental data in GitHub. The original CSV checksums were verified in Section A.


In [ ]:
analysis_script = ROOT / 'scripts' / 'rebuild_phase1_artifacts.py'
subprocess.run([sys.executable, str(analysis_script), '--root', str(ROOT)], check=True)
print((ROOT/'reports'/'quality_summary.json').read_text())


In [ ]:
best = pd.read_csv(ROOT / 'tables' / 'formal_best_comparison.csv')
cols = [
    'p_label', 'chebyshev_configuration', 'chebyshev_median_ms',
    'newton_configuration', 'newton_median_ms',
    'bisection_configuration', 'bisection_median_ms',
    'cheb_over_newton'
]
display(best[cols].round(4))


### Measurement definition

The accuracy filter uses

$$
E_\infty=\max_i\frac{|\widehat x_i-x_i^{\mathrm{ref}}|}{\max\{|y_i|,\tau_p\}}\le 10^{-6},
$$

along with finite outputs and zero support mismatches for the derived summary. `max_active_stationarity_residual` is a **separate, unnormalized** diagnostic; it is not the accuracy filter.

The reference is float64 bracketed bisection (75 iterations), cross-checked against independent float64 Newton on a subset. The current protocol records medians/standard deviations but **not** the individual timing repetitions.

### Data-quality audit

In [ ]:
quality = pd.read_csv(ROOT / 'reports' / 'formal_quality_checks.csv')
display(quality[['p_label','n_configurations','n_pass_accuracy',
                 'n_nonfinite','total_support_mismatches',
                 'max_reference_crosscheck_norm_error','commit']])

all_rows = pd.read_csv(ROOT / 'tables' / 'formal_all_configurations.csv')
assert len(all_rows) == 112
assert (all_rows['n_timing'] == 1_000_000).all()
assert (all_rows['dtype'] == 'float32').all()
assert (all_rows['support_mismatch_count'] == 0).all()
print('Formal rows validated:', len(all_rows))


## C · Figures for the formal runs

Figures are generated directly from the CSVs and saved as both 220-dpi PNG and vector PDF. The stars in the six-panel figure indicate the **fastest passing** configuration within the tested grid.

In [ ]:
for filename in ['best_runtime.png','cheb_newton_time_ratio.png','accuracy_runtime_six_panel.png']:
    print(filename)
    display(Image(filename=str(ROOT / 'figures' / filename)))


## D · Explore an individual exponent and export the tables

Change `P_TO_EXPLORE` to inspect all recorded Chebyshev/Newton/Bisection configurations for one exponent. The selection does not alter the raw data.

In [ ]:
P_TO_EXPLORE = '0.999'  # '1/3', '2/5', '3/4', '0.9', '0.99', or '0.999'
sub = all_rows[all_rows['p_label'].astype(str) == P_TO_EXPLORE].copy()
fields = ['configuration','median_ms','std_ms','max_norm_error',
          'max_active_stationarity_residual','meets_accuracy_target',
          'sampled_float64_branch_error']
display(sub[fields].sort_values('median_ms').reset_index(drop=True))


---
## E · Optional: rerun formal GPU benchmarks at the recorded code commit

**Disabled by default.** Only turn on `RUN_BENCHMARKS=True` if you explicitly want a *new* CUDA measurement. GitHub dataset retrieval in Section A does not require a GPU.

The recorded formal *benchmark source* is commit `7a1e32b`, which predates the data-archive commits. We use a **separate solver checkout** for optional reruns so that the data archive remains intact. The exact parameter grids are reconstructed from the archived CSV metadata. Reruns are saved under the isolated workspace at `data/rerun/`, never under `data/formal/`.

The GPU model, PyTorch version, elapsed times and source revision should be compared with the historical A100 measurements, not assumed identical.


In [ ]:
RUN_BENCHMARKS = False  # Change to True ONLY to launch new GPU benchmarking
SELECTED_RUNS = ['p0333']  # choose from p0333, p0400, p0750, p0900, p0990, p0999
PINNED_COMMIT = '7a1e32b'
BENCHMARK_REPO_DIR = Path('/content/Fast_Lp_Prox_Benchmark_Code') if Path('/content').is_dir() \
                     else Path.home() / '.cache' / 'Fast_Lp_Prox_Benchmark_Code'

if RUN_BENCHMARKS:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA is unavailable; in Colab select Runtime → Change runtime type → GPU.')
    print('Current GPU:', torch.cuda.get_device_name(0))
    if not (BENCHMARK_REPO_DIR / '.git').is_dir():
        if BENCHMARK_REPO_DIR.exists():
            raise RuntimeError(f'Non-Git directory exists: {BENCHMARK_REPO_DIR}')
        BENCHMARK_REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL,
                        str(BENCHMARK_REPO_DIR)], check=True)
    # Full commit must be available even if an old checkout was shallow.
    subprocess.run(['git', 'fetch', 'origin', REPO_BRANCH], cwd=BENCHMARK_REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', '--detach', PINNED_COMMIT], cwd=BENCHMARK_REPO_DIR, check=True)
    commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=BENCHMARK_REPO_DIR,
                                     text=True).strip()
    if not commit.startswith(PINNED_COMMIT):
        raise RuntimeError(f'Unexpected code commit: {commit}')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev]'],
                   cwd=BENCHMARK_REPO_DIR, check=True)
    subprocess.run([sys.executable, '-m', 'pytest', '-q',
                    'tests/test_revision_accuracy_runtime.py'],
                   cwd=BENCHMARK_REPO_DIR, check=True)
    print('Reproduction source verified:', commit)
else:
    print('Reruns disabled. Only archived data are analyzed.')


In [ ]:
# Build the exact candidate grids from the archived formal CSV metadata.
# This avoids copying degree/iteration lists by hand.
if RUN_BENCHMARKS:
    specs = json.loads((ROOT/'reports'/'formal_parameter_grids.json').read_text())
    specs = {spec['tag']:spec for spec in specs}
    permitted = set(specs)
    if not set(SELECTED_RUNS).issubset(permitted):
        raise ValueError(f'Unknown run ID; choose from {sorted(permitted)}')
    (ROOT/'data'/'rerun').mkdir(exist_ok=True)
    for run_id in SELECTED_RUNS:
        item = specs[run_id]
        info = item['common']
        target = ROOT/'data'/'rerun'/item['source_file'].replace('.csv','_rerun.csv')
        if target.exists():
            print('SKIP existing rerun (no overwrite):', target)
            continue
        argv = [
            sys.executable,'scripts/benchmark_accuracy_runtime.py',
            '--device','cuda','--dtype','float32',
            '--p',item['p_label'],'--lam',str(info['lambda']),
            '--n',str(info['n_timing']),
            '--active-ratio',str(info['active_ratio_requested']),
            '--ymax-factor',str(info['ymax_factor']),
            '--seed',str(info['seed']),
            '--cheb-degrees',','.join(map(str,item['cheb_degrees'])),
            '--newton-iters-grid',','.join(map(str,item['newton_iterations'])),
            '--bisect-iters-grid',','.join(map(str,item['bisection_iterations'])),
            '--n-fit',str(item['n_fit']),
            '--n-val',str(info['validation_n']),
            '--warmup',str(info['warmup']),
            '--repeats',str(info['repeats']),
            '--target-norm-error',str(info['accuracy_target']),
            '--output',str(target),
        ]
        print('Re-running',item['p_label'], 'with',len(item['cheb_degrees']),
              'Chebyshev degrees,',len(item['newton_iterations']),'Newton steps,',
              len(item['bisection_iterations']),'bisection steps')
        subprocess.run(argv,cwd=BENCHMARK_REPO_DIR,check=True)
    print('New rerun CSV files:',sorted(p.name for p in (ROOT/'data'/'rerun').glob('*.csv')))


## E.2 · Compare any independent reruns with the archived baseline

The rerun timings are shown **separately**. They are not silently merged with the original A100 runs.

In [ ]:
rerun_dir = ROOT/'data'/'rerun'
if rerun_dir.is_dir() and list(rerun_dir.glob('*.csv')):
    comparisons = []
    for new_file in sorted(rerun_dir.glob('*_rerun.csv')):
        original_file = FORMAL_DIR/new_file.name.replace('_rerun.csv','.csv')
        if not original_file.exists():
            continue
        base = pd.read_csv(original_file)
        new = pd.read_csv(new_file)
        join = base[['configuration','median_ms','max_norm_error']].merge(
            new[['configuration','median_ms','max_norm_error']],
            on='configuration',suffixes=('_archived','_rerun'))
        join['source_file'] = original_file.name
        join['rerun_over_archived'] = join['median_ms_rerun']/join['median_ms_archived']
        comparisons.append(join)
    if comparisons:
        display(pd.concat(comparisons,ignore_index=True))
else:
    print('No reruns yet. Archived formal data remain unchanged.')


---
## F · Export reproducible Phase 1 ZIP

Export a self-contained copy of the **isolated workspace**: original, verified raw CSVs; rebuilt summary tables; figures; quality reports; public analysis scripts; and any explicitly requested `data/rerun/` files. No files are added to GitHub by running this notebook.

The export deliberately includes only `revision/phase1` archive materials, **not** arbitrary files from the solver checkout or private manuscript/reviewer directories.


In [ ]:
from datetime import datetime, timezone
export_directory = Path('/content') if Path('/content').is_dir() else Path.cwd()
EXPORT = export_directory / ('FastLp_Phase1_Rebuilt_' +
                             datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '.zip')
with zipfile.ZipFile(EXPORT, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=8) as archive:
    for file in sorted(ROOT.rglob('*')):
        if not file.is_file() or '.ipynb_checkpoints' in file.parts or '__pycache__' in file.parts:
            continue
        archive.write(file, file.relative_to(ROOT))
print('Exported:', EXPORT, 'size:', EXPORT.stat().st_size, 'bytes')
try:
    from google.colab import files
    files.download(str(EXPORT))
except ImportError:
    print('Local export ready:', EXPORT)


### Limits and the next experimental phase

The six recorded speed-vs-accuracy sweeps compare current PyTorch prox implementations on a sampled input distribution. They do **not** provide a whole-domain certified error, a deterministic threshold-neighborhood test, a comparison of fused GPU kernels, a multi-seed/interleaved timing study, or end-to-end ℓp-Lasso convergence experiments.

The data archive on `revision/major-experiments` and the historical solver benchmark code at `7a1e32b` have distinct Git revision roles. Archived outputs are never overwritten by optional reruns. Future threshold and timing reliability studies should be recorded as separate artifacts (Phase 2).
